# 01 · Exploring workload, ACWR and athlete baselines

Interactive companion to `docs/methodology.md`. **All data is synthetic.** Run from the repository root (`jupyter lab notebooks/`) after `pip install -r requirements-dev.txt jupyterlab`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MLFLOW_DISABLED', '1')
import pandas as pd
from src.config import load_config
from src.data.repository import load_dataset
from src.data.validation import validate_dataset
cfg = load_config()
ds = load_dataset(cfg, prefer_database=False)
print(validate_dataset(ds.athletes, ds.sessions, ds.absences).summary())
ds.sessions.head()

## Session demands by type and position

In [ ]:
ds.sessions.groupby('session_type')[['duration_minutes','total_distance','high_speed_running_distance','sprint_distance','player_load','perceived_exertion']].median().round(0)

In [ ]:
m = ds.sessions[ds.sessions.match_or_training == 'match']
m.groupby('position')[['total_distance','high_speed_running_distance','sprint_distance','accelerations','max_speed']].mean().round(0)

## Leakage-free modelling table

In [ ]:
from src.features import build_modeling_table
table = build_modeling_table(ds, cfg)
lab = table[table.eligible & table.observable_7d]
print(len(lab), 'labelled athlete-days, event rate', round(lab.availability_risk_7d.mean(), 4))

## ACWR: descriptive association (not causation)

Coupled rolling vs EWMA vs uncoupled. Note how sparse the high-ratio bands are and how wide the uncertainty is.

In [ ]:
bins = [0, .8, 1, 1.3, 1.5, 2, 99]
for col in ['acwr_total_load', 'acwr_total_load_uncoupled', 'ewma_acwr_total_load']:
    print(lab.groupby(pd.cut(lab[col], bins), observed=False).availability_risk_7d.agg(['mean', 'size']).round(3), '\n')

## Athlete-specific baselines

The same absolute sprint volume means different things for different athletes.

In [ ]:
a = lab.xs('A172', level='athlete_id')[['sprint_distance_7d','sprint_distance_baseline_weekly','sprint_distance_vs_baseline_pct','sprint_distance_vs_baseline_z']]
a.tail(14).round(1)

## A single explained prediction and a what-if simulation

In [ ]:
from src.service import RiskService
svc = RiskService(cfg, dataset=ds)
out = svc.predict('A172')
print(out['summary'])
for f in out['top_risk_factors']: print('+', f['explanation'])
for f in out['protective_factors']: print('-', f['explanation'])

In [ ]:
sim = svc.simulate('A172', scenarios=[{'sprint_load_pct': 20}, {'sprint_load_pct': -20}, {'rest_day': True}])
pd.DataFrame([{'scenario': s['name'], 'risk': s['risk_probability'], 'delta': s['delta_probability']} for s in sim['scenarios']])